In [1]:
import numpy as np
from helpers import *
from implementations import *

Import the data

In [2]:
# You have to change the path for it to work
data_path = r'C:\Users\natha\Documents\EPFL\Cours_MA1\ML\ML_course\projects\project1 - withGit\data\dataset'

In [3]:
x_train, x_test, y_train, train_ids, test_ids, headers_train = load_csv_data(data_path, sub_sample=False)

test


In [21]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)

(328135, 321)
(109379, 321)
(328135,)


Convert -1 into 0 in y

In [22]:
y_train[y_train == -1] = 0

# Check if all values are either 0 or 1
if np.all((y_train == 0) | (y_train == 1)):
    print("All values are either 0 or 1.")
else:
    raise ValueError("The array contains values other than 0 or 1.")

All values are either 0 or 1.


Remove columns with Nan

In [28]:
# Function to find the count of missing values in each columns
def find_missing_values(data, headers=None):
    num_rows, num_cols = data.shape
    missing_count = np.zeros(num_cols, dtype=int)  
    columns = np.linspace(0,num_cols, num_cols+1)
    
    for col in range(num_cols):
        # Count the missing values
        missing_count[col] = np.sum(np.isnan(data[:, col]))
    if headers : 
        # Returning only the columns with missing values
        missing_info = {headers[col]: missing_count[col] for col in range(num_cols) if missing_count[col] > 0}
    else : 
        missing_info = {columns[col]: missing_count[col] for col in range(num_cols) if missing_count[col] > 0}
    return missing_info

In [34]:
# Function to remove columns with more than 1/3 of missing values
def remove_high_missing_columns(data):
    num_rows, num_cols = data.shape
    threshold = 0
    missing_count = find_missing_values(data) 

    # Create a mask for columns to keep
    columns_to_keep = [col for col in range(num_cols) if missing_count.get(col, 0) <= threshold]

    # Return a new array with only the columns that meet the criteria
    return data[:, columns_to_keep]

In [35]:
sliced_x_train = remove_high_missing_columns(x_train)

In [36]:
sliced_x_train.shape

(328135, 82)

In [39]:
column_with_nan = find_missing_values(sliced_x_train)

In [40]:
print(column_with_nan)

{}


Standardize data to avoid issues with large numbers

In [41]:
def standardize(x):
    """Stadartize the input data x

    Args:
        x: numpy array of shape=(num_samples, num_features)

    Returns:
        standartized data, shape=(num_samples, num_features)

    >>> standardize(np.array([[1, 2], [3, 4], [5, 6]]))
    array([[-1.22474487, -1.22474487],
           [ 0.        ,  0.        ],
           [ 1.22474487,  1.22474487]])
    """
    # ***************************************************
    means=np.mean(x, axis=0)
    stds=np.std(x, axis=0)
    std_data=(x-means)/stds
    # ***************************************************

    return std_data

In [46]:
x_train = standardize(sliced_x_train)

Split the data

In [47]:
def split_data(x, y, ratio, seed=1):
    """
    split the dataset based on the split ratio. If ratio is 0.8
    you will have 80% of your data set dedicated to training
    and the rest dedicated to testing. If ratio times the number of samples is not round
    you can use np.floor. Also check the documentation for np.random.permutation,
    it could be useful.

    Args:
        x: numpy array of shape (N,), N is the number of samples.
        y: numpy array of shape (N,).
        ratio: scalar in [0,1]
        seed: integer.

    Returns:
        x_tr: numpy array containing the train data.
        x_te: numpy array containing the test data.
        y_tr: numpy array containing the train labels.
        y_te: numpy array containing the test labels.

    >>> split_data(np.arange(13), np.arange(13), 0.8, 1)
    (array([ 2,  3,  4, 10,  1,  6,  0,  7, 12,  9]), array([ 8, 11,  5]), array([ 2,  3,  4, 10,  1,  6,  0,  7, 12,  9]), array([ 8, 11,  5]))
    """
    # set seed
    np.random.seed(seed)
    # ***************************************************
    indices = np.random.permutation(x.shape[0])
    x_shuffled = x[indices]
    y_shuffled = y[indices]
    
    first_index_te = int(np.floor(x.shape[0] * ratio))
    x_tr = x_shuffled[:first_index_te]
    x_te = x_shuffled[first_index_te:]
    y_tr = y_shuffled[:first_index_te]
    y_te = y_shuffled[first_index_te:]
    return x_tr, x_te, y_tr, y_te
    # ***************************************************

In [48]:
x_tr, x_val, y_tr, y_val = split_data(x_train, y_train, 0.8, seed=1)
print(x_tr.shape)
print(x_val.shape)
print(y_tr.shape)
print(y_val.shape)

(262508, 82)
(65627, 82)
(262508,)
(65627,)


Train the model

In [55]:
initial_w = np.zeros(x_tr.shape[1])
max_iters = 1000
gamma = 0.01

w_opt, loss_opt = logistic_regression(y_tr, x_tr, initial_w, max_iters, gamma)

Current iteration=0, loss=0.6931471805599448
Current iteration=100, loss=0.6798311686980676
Current iteration=200, loss=0.677077772792685
Current iteration=300, loss=0.6759188144666871
Current iteration=400, loss=0.6753015335261586
Current iteration=500, loss=0.6749383536742685
Current iteration=600, loss=0.6747117674652988
Current iteration=700, loss=0.6745643661179866
Current iteration=800, loss=0.6744650213813728
Current iteration=900, loss=0.6743957238318639
loss=0.6743456230246271


In [56]:
print(w_opt)

[ 0.00197162  0.00235892 -0.00088374 -0.00076322 -0.00501345 -0.00640743
 -0.00061009 -0.00160796 -0.00160796 -0.02370442 -0.01031219 -0.00075669
 -0.00909962 -0.07841666 -0.01337256 -0.01545327 -0.05129871 -0.02086564
 -0.03647051 -0.10649079 -0.00718399 -0.01665943  0.01696727  0.07568913
  0.00035039  0.00198082 -0.00275991  0.00070032  0.01208939 -0.00147407
  0.12952601  0.03678447  0.09290622 -0.01637887  0.01958918  0.01548066
 -0.01762125 -0.00394337 -0.00242324  0.00303807 -0.00176953  0.00016659
 -0.001899    0.05138148  0.03252825  0.05401973  0.01957945 -0.00273561
  0.0014241  -0.0122638  -0.0261104  -0.03328646  0.0123537   0.02863627
 -0.01079563 -0.00589586 -0.00017318 -0.00346351 -0.00172792 -0.00311317
  0.00088932  0.00120968  0.00364643  0.00237272 -0.00465223  0.00446318
 -0.00043014 -0.00151203  0.00617434 -0.0214953  -0.01619565 -0.00722545
  0.01253724 -0.00558596  0.00528968  0.00252888 -0.00843896 -0.00080471
  0.0119057  -0.00624027  0.00168196  0.00305944]


Apply the model to the validation set

In [57]:
g = x_val @ w_opt
g.shape

(65627,)

In [58]:
limit = 0.5

g[g <= limit] = 0
g[g > limit] = 1

# Check if all values are either 0 or 1
if np.all((g == 0) | (g == 1)):
    print("All values are either 0 or 1.")
else:
    raise ValueError("The array contains values other than 0 or 1.")

All values are either 0 or 1.


Check our accuracy and F1

In [59]:
accuracy = np.mean(y_val == g)

print("Accuracy:", accuracy)

Accuracy: 0.8792874883813064


In [60]:
TP = np.sum((y_val == 1) & (g == 1))
FP = np.sum((y_val == 0) & (g == 1))
FN = np.sum((y_val == 1) & (g == 0))

# Calculate Precision and Recall
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

# Calculate F1 Score
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)

Precision: 0.33406787130894666
Recall: 0.4015539466713756
F1 Score: 0.36471531676022456
